# 1 - Imports and Path Setup

In [1]:
import os
import re
import json
import collections
import sys

In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'
VOCAB_DIR    = os.path.join(PROJECT_ROOT, 'vocab')

In [3]:
sys.path.insert(0, PROJECT_ROOT)

In [4]:
SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>', '<unk>']
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

In [5]:
print(f"   VOCAB_DIR : {VOCAB_DIR}")
print(f"   Special tokens: {SPECIAL_TOKENS}")
print(f"   PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}, UNK={UNK_ID}")

   VOCAB_DIR : /content/drive/MyDrive/Transformer_Mine/vocab
   Special tokens: ['<pad>', '<bos>', '<eos>', '<unk>']
   PAD=0, BOS=1, EOS=2, UNK=3


# 2 - Text Normalisation

Before BPE, we apply light text normalisation:

1. **Lowercase (optional)** — controlled by `DO_LOWERCASE`.
2. **Punctuation spacing** — ensure punctuation has spaces around it so `"dog."` becomes `"dog ."` and each punctuation mark can be treated as a separate unit.
3. **Collapse multiple spaces** — convert consecutive spaces into a single space.
4. **Strip whitespace** — remove leading and trailing whitespace.

We do **not** apply aggressive cleaning such as HTML stripping because Stage A and Tatoeba are already relatively clean.

> **Note:** `DO_LOWERCASE = False` for EN-DE to preserve German capitalisation, since nouns are capitalised in German.

In [6]:
IS_LOWERCASE = False

In [7]:
def normalise(text: str) -> str:
    """
    Light normalisation for EN-DE translation data.

    Args:
        text: raw input string

    Returns:
        normalised string
    """
    # 1. Lowercase
    if IS_LOWERCASE:
        text = text.lower()

    # 2. Punctuation spacing
    text = re.sub(r'([\.,!?;:\(\)\[\]{}\"\'\-/])', r' \1 ', text)

    # 3. Collapse multiple whitespace into single space
    text = re.sub(r'\s+', ' ', text)

    # 4. Strip Whitespaces
    text = text.strip()

    return text

In [8]:
test_cases = [
    ("I am a student.", "I am a student ."),
    ("She is a teacher!", "She is a teacher !"),
    ("Er ist ein Arzt.", "Er ist ein Arzt ."),
    ("  extra   spaces  ", "extra spaces"),
    ("word,word", "word , word"),
]

In [9]:
print("Normalisation tests:")
all_pass = True
for raw, expected in test_cases:
    result = normalise(raw)
    status = "<GOOD>" if result == expected else "X"
    if result != expected:
        all_pass = False
    print(f"  {status}  '{raw}'")
    print(f"       → '{result}'")
    if result != expected:
        print(f"       expected: '{expected}'")

if all_pass:
    print("\n✅ All normalisation tests passed.")
else:
    print("\n❌ Some normalisation tests failed. Check regex.")

Normalisation tests:
  <GOOD>  'I am a student.'
       → 'I am a student .'
  <GOOD>  'She is a teacher!'
       → 'She is a teacher !'
  <GOOD>  'Er ist ein Arzt.'
       → 'Er ist ein Arzt .'
  <GOOD>  '  extra   spaces  '
       → 'extra spaces'
  <GOOD>  'word,word'
       → 'word , word'

✅ All normalisation tests passed.


# 3 - Word Frequency Counter

BPE starts with a **word-level frequency count** of the corpus.

Each word is represented as a sequence of characters with a special end-of-word marker `</w>`.

For example:

```text
"student" → ('s', 't', 'u', 'd', 'e', 'n', 't', '</w>')

In [10]:
END_OF_WORD = '</w>'

In [11]:
def text_to_word_frequencies(lines: list, normalise_func = normalise) -> collections.Counter:
    """
    Build a word frequency Counter from a list of text lines.

    Each word is stored as a tuple of characters with </w> appended
    to the last character, representing the end-of-word boundary.

    Args:
        lines       : list of raw strings
        normalise_fn: normalisation function to apply first

    Returns:
        Counter mapping word-tuple → frequency
    """

    freq = collections.Counter()
    for line in lines:
        line = normalise_func(line)

        words = line.split()
        for word in words:
            chars = tuple(list(word[:-1]) + [word[-1] + END_OF_WORD])
            freq[chars] += 1

    return freq

In [12]:
def get_pair_frequencies(
    word_freqs: collections.Counter
) -> collections.Counter:
    """
    Count all adjacent character-pair frequencies across all words.

    For each word (represented as a tuple of symbols), count how
    often each adjacent pair of symbols appears, weighted by the
    word's frequency in the corpus.

    Args:
        word_freqs: Counter of word-tuple → frequency

    Returns:
        Counter of (symbol_a, symbol_b) → total frequency

    Example:
        word_freqs = {('s','t','u','d','e','n','t</w>'): 3}
        pair_freqs = {('s','t'): 3, ('t','u'): 3, ...}
    """

    pairs = collections.Counter()
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq

    return pairs

This weighted sum allows Byte-Pair Encoding (BPE) to identify which character pairs are most common across the entire corpus, so it knows which pair to merge into a new vocabulary token.

In [13]:
def merge_pair(
    pair: tuple,
    word_freqs: collections.Counter,
) -> collections.Counter:
    """
    Merge all occurrences of a symbol pair in every word.

    Replaces every adjacent occurrence of (a, b) with the merged
    symbol 'ab' across all words in the vocabulary.

    Args:
        pair      : (symbol_a, symbol_b) to merge
        word_freqs: current word frequency Counter

    Returns:
        new Counter with the pair merged everywhere

    Example:
        pair = ('e', 's</w>')
        ('i','n','t','e','r','e','s','</w>') → ('i','n','t','e','r','es</w>')
    """
    new_freqs = collections.Counter()
    a, b      = pair
    merged    = a + b

    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                new_word.append(merged)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_freqs[tuple(new_word)] += freq

    return new_freqs

In [14]:
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Change working directory directly to your folder
%cd /content/drive/MyDrive/Transformer_Mine

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Transformer_Mine


In [15]:
from data_pipeline import (
    BOS_ID,
    EOS_ID,
    PAD_ID,
    UNK_ID,
    build_tf_dataset,
    create_decoder_mask,
    create_look_ahead_mask,
    create_padding_mask,
    encode_and_wrap,
    load_raw_pairs,
    pad_sequence,
)

In [16]:
STAGE_A_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_a')
src_a, tgt_a = load_raw_pairs(
    os.path.join(STAGE_A_DIR, 'train.en'),
    os.path.join(STAGE_A_DIR, 'train.de'),
)

all_lines_a = src_a + tgt_a
word_freqs_a = text_to_word_frequencies(all_lines_a)

print(f"Stage A word types (unique words): {len(word_freqs_a)}")
print("\nTop 10 most frequent words (as character tuples):")
for word, freq in word_freqs_a.most_common(10):
    readable = ''.join(word).replace(END_OF_WORD, '')
    print(f"  {readable:<20} freq={freq}  repr={word}")

pair_freqs_a = get_pair_frequencies(word_freqs_a)
print(f"\nTop 5 most frequent character pairs:")
for pair, freq in pair_freqs_a.most_common(5):
    print(f"  {pair}  →  freq={freq}")

Stage A word types (unique words): 81

Top 10 most frequent words (as character tuples):
  .                    freq=40  repr=('.</w>',)
  is                   freq=7  repr=('i', 's</w>')
  ist                  freq=7  repr=('i', 's', 't</w>')
  a                    freq=6  repr=('a</w>',)
  Sie                  freq=6  repr=('S', 'i', 'e</w>')
  I                    freq=5  repr=('I</w>',)
  She                  freq=5  repr=('S', 'h', 'e</w>')
  Ich                  freq=5  repr=('I', 'c', 'h</w>')
  He                   freq=4  repr=('H', 'e</w>')
  Er                   freq=4  repr=('E', 'r</w>')

Top 5 most frequent character pairs:
  ('h', 'e</w>')  →  freq=9
  ('c', 'h</w>')  →  freq=8
  ('e', 'i')  →  freq=8
  ('i', 's</w>')  →  freq=7
  ('h', 'a')  →  freq=7


# 4 - BPE Training Algorithm

Implements the BPE algorithm from:

**Sennrich et al. (2016), "Neural Machine Translation of Rare Words with Subword Units"**

## Algorithm

1. Start with a **character-level vocabulary**.
2. Count all **adjacent symbol pairs**.
3. Merge the **most frequent pair** into a new symbol.
4. Repeat the process for `num_merges` iterations.
5. The **ordered list of merge rules** defines the tokenizer. These rules are what get saved and reloaded.

The original paper uses a **shared vocabulary** for the source and target languages.

I followed the same convention by training BPE on the **combined source + target corpus**.

## Parameters

- **Stage A:** Small `num_merges` because the dataset contains only approximately 40 unique words.
- **Stage B:** `num_merges = vocab_size - len(SPECIAL_TOKENS) - n_chars`, typically with `vocab_size = 8000`.
- **Stage D:** `vocab_size ≈ 37000`, following the vocabulary size used in the original paper.

In [17]:
def train_bpe(
    lines      : list,
    vocab_size : int,
    normalise_fn = normalise,
    verbose    : bool = True,
) -> tuple:
    """
    Train BPE on a list of text lines.

    Args:
        lines      : combined source + target lines
        vocab_size : target vocabulary size INCLUDING special tokens
        normalise_fn: text normalisation function
        verbose    : print progress every 1000 merges

    Returns:
        merges    : ordered list of (symbol_a, symbol_b) merge rules
        vocab     : final set of all symbols (characters + merged)
        word_freqs: final word frequency Counter after all merges

    Notes:
        - Special tokens are added separately (not learned by BPE).
        - vocab_size is a TARGET — BPE stops when:
            len(SPECIAL_TOKENS) + len(base_chars) + len(merges)
            >= vocab_size
          or when no pairs remain.
    """
    # ── Step 1: Build initial character-level word frequencies ──
    word_freqs = text_to_word_frequencies(lines, normalise_fn)

    # ── Step 2: Initial vocabulary = all unique characters ──────
    base_vocab = set()
    for word in word_freqs:
        for symbol in word:
            base_vocab.add(symbol)

    if verbose:
        print(f"Initial character vocabulary size : {len(base_vocab)}")
        print(f"Unique word types in corpus       : {len(word_freqs)}")
        print(f"Target vocabulary size            : {vocab_size}")
        n_special = len(SPECIAL_TOKENS)
        n_merges_target = vocab_size - n_special - len(base_vocab)
        print(f"Merges needed                     : "
              f"max({n_merges_target}, 0)")
        print()

    # ── Step 3: BPE merge loop ───────────────────────────────────
    merges= []   # ordered list of merge rules
    vocab = set(base_vocab)

    n_special = len(SPECIAL_TOKENS)
    merges_target  = max(0, vocab_size - n_special - len(base_vocab))

    for i in range(merges_target):
        pair_freqs = get_pair_frequencies(word_freqs)

        if not pair_freqs:
            if verbose:
                print(f"  No more pairs to merge at step {i}. Stopping.")
            break

        # Find the most frequent pair
        best_pair = max(pair_freqs, key=pair_freqs.get)
        best_freq = pair_freqs[best_pair]

        if best_freq < 2:
            if verbose:
                print(f"  Best pair frequency < 2 at step {i}. Stopping.")
            break

        # Merge
        merges.append(best_pair)
        merged_symbol = best_pair[0] + best_pair[1]
        vocab.add(merged_symbol)
        word_freqs = merge_pair(best_pair, word_freqs)

        if verbose and (i + 1) % max(1, merges_target // 10) == 0:
            print(f"  Merge {i+1:>5}/{merges_target}: "
                  f"'{best_pair[0]}' + '{best_pair[1]}' → "
                  f"'{merged_symbol}'  (freq={best_freq})")

    total_vocab_size = n_special + len(vocab)
    if verbose:
        print(f"\n✅ BPE training complete.")
        print(f"   Merges learned    : {len(merges)}")
        print(f"   Symbol vocab size : {len(vocab)}")
        print(f"   Total vocab size  : {total_vocab_size}  "
              f"(+{n_special} special tokens)")

    return merges, vocab, word_freqs



In [19]:
# ── Train on Stage A (tiny, instant) ────────────────────────
print("Training BPE on Stage A data …")
print("=" * 50)

merges_a, vocab_a, word_freqs_a_final = train_bpe(
    lines = src_a + tgt_a,
    vocab_size = 200,   # tiny for Stage A
    verbose = True,
)

print(f"\nSample merge rules (first 10):")
for i, (a, b) in enumerate(merges_a[:10]):
    print(f"  {i+1:>3}. '{a}' + '{b}' → '{a+b}'")

Training BPE on Stage A data …
Initial character vocabulary size : 56
Unique word types in corpus       : 81
Target vocabulary size            : 200
Merges needed                     : max(140, 0)

  Merge    14/140: 'tu' + 'd' → 'tud'  (freq=4)
  Merge    28/140: 'li' + 'k' → 'lik'  (freq=3)
  Merge    42/140: 'ch' + 'er</w>' → 'cher</w>'  (freq=2)
  Merge    56/140: 'ca' + 't</w>' → 'cat</w>'  (freq=2)
  Merge    70/140: 'a' + 'u' → 'au'  (freq=2)
  Best pair frequency < 2 at step 77. Stopping.

✅ BPE training complete.
   Merges learned    : 77
   Symbol vocab size : 133
   Total vocab size  : 137  (+4 special tokens)

Sample merge rules (first 10):
    1. 'h' + 'e</w>' → 'he</w>'
    2. 'c' + 'h</w>' → 'ch</w>'
    3. 'e' + 'i' → 'ei'
    4. 'i' + 's</w>' → 'is</w>'
    5. 'h' + 'a' → 'ha'
    6. 'i' + 'e</w>' → 'ie</w>'
    7. 'i' + 's' → 'is'
    8. 'is' + 't</w>' → 'ist</w>'
    9. 'S' + 'ie</w>' → 'Sie</w>'
   10. 'e' + 'n' → 'en'


# 5 - BPE Encoder — Apply Merges to New Text

Given a trained list of **BPE merge rules**, encode any new text by applying the learned merges in their original order.

## Algorithm for Encoding a Single Word

1. Split the word into individual characters and append `</w>` to the final character.

2. Iterate through each merge rule in the learned order:
   - Find occurrences of the corresponding symbol pair in the word.
   - Merge each occurrence into a single subword symbol.

3. Return the resulting list of **subword symbols**.

Conceptually:

```text
New Word
   ↓
Characters + </w>
   ↓
Apply Merge Rule 1
   ↓
Apply Merge Rule 2
   ↓
Apply Merge Rule 3
   ↓
...
   ↓
Final Subword Symbols

In [20]:
def encode_word(word: str, merge_rules: list) -> list:
    """
    Apply BPE merge rules to a single word.

    Args:
        word       : a single whitespace-tokenised word, e.g. "student"
        merge_rules: ordered list of (symbol_a, symbol_b) tuples

    Returns:
        list of subword symbols, e.g. ['stud', 'ent</w>']
    """
    if (len(word) == 0): return []

    symbols = list(word[:-1]) + [word[-1] + END_OF_WORD]

    for (a, b) in merge_rules:
        merged = a + b
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols

    return symbols

In [26]:
def encode_line(
    text       : str,
    merge_rules: list,
    token2id   : dict,
    normalise_fn = normalise,
    unk_id     : int = UNK_ID,
) -> list:
    """
    Encode a full line of text into a list of integer token IDs.

    Process:
        raw text → normalise → split words → BPE encode each word
        → look up IDs → return list of ints

    Args:
        text       : raw input string
        merge_rules: BPE merge rules
        token2id   : symbol → integer ID mapping
        normalise_fn: normalisation function
        unk_id     : ID for unknown symbols

    Returns:
        list of integer token IDs (no BOS/EOS — added separately)
    """
    text = normalise_fn(text)
    ids= []

    for word in text.split():
        subwords = encode_word(word, merge_rules)
        for sw in subwords:
            ids.append(token2id.get(sw, unk_id))

    return ids

In [27]:
def decode_ids(
    ids: list,
    id2token: dict,
    pad_id: int = PAD_ID,
    bos_id : int = BOS_ID,
    eos_id: int = EOS_ID,
) -> str:
    """
    Decode a list of token IDs back to a human-readable string.

    Process:
        IDs → symbols → join → remove </w> → clean spaces

    Args:
        ids: list of integer token IDs
        id2token: integer ID → symbol mapping
        pad_id, bos_id, eos_id: special IDs to skip

    Returns:
        decoded string

    Example:
        [1, 15, 22, 8, 2] → "Ich bin ."
    """
    skip = {pad_id, bos_id, eos_id}
    tokens = [id2token.get(i, '<unk>') for i in ids if i not in skip]

    # Join subwords: </w> marks end of word → replace with space
    text = ''.join(tokens)
    text = text.replace(END_OF_WORD, ' ')
    text = text.strip()

    return text


In [28]:
# Build token2id / id2token from Stage A BPE
# Special tokens first (fixed IDs 0-3), then BPE symbols
all_symbols_a = SPECIAL_TOKENS + sorted(vocab_a)
token2id_a    = {s: i for i, s in enumerate(all_symbols_a)}
id2token_a    = {i: s for s, i in token2id_a.items()}

print(f"Vocabulary size (Stage A): {len(token2id_a)}")

Vocabulary size (Stage A): 137


In [29]:
# Encode/decode round-trip tests
test_sentences = [
    "I am a student .",
    "She is a teacher .",
    "Ich bin ein Student .",
    "Sie ist eine Lehrerin .",
]

print("\nRound-trip encode → decode tests:")
print(f"{'Input':<35} {'Decoded':<35} {'Match'}")
print("-" * 80)

all_pass = True
for sent in test_sentences:
    ids     = encode_line(sent, merges_a, token2id_a)
    decoded = decode_ids(ids, id2token_a)

    # Compare normalised versions (punctuation spacing may differ)
    match   = normalise(sent) == normalise(decoded)
    status  = "✅" if match else "X"
    if not match:
        all_pass = False
    print(f"{sent:<35} {decoded:<35} {status}")

if all_pass:
    print("\n✅ All round-trip tests passed.")
else:
    print("\n⚠️  Some round-trips differ — check merge rules or </w> handling.")

# subword segmentation
print("\nSubword segmentation examples:")
for word in ["student", "teacher", "Lehrerin", "Studenten"]:
    subwords = encode_word(word, merges_a)
    print(f"  '{word}' → {subwords}")


Round-trip encode → decode tests:
Input                               Decoded                             Match
--------------------------------------------------------------------------------
I am a student .                    I am a student .                    ✅
She is a teacher .                  She is a teacher .                  ✅
Ich bin ein Student .               Ich bin ein Student .               ✅
Sie ist eine Lehrerin .             Sie ist eine Lehrerin .             ✅

✅ All round-trip tests passed.

Subword segmentation examples:
  'student' → ['studen', 't</w>']
  'teacher' → ['te', 'a', 'cher</w>']
  'Lehrerin' → ['L', 'eh', 're', 'ri', 'n</w>']
  'Studenten' → ['Studen', 'te', 'n</w>']


#6 - BPETokenizer Class

Wraps the complete BPE implementation into a clean and reusable `BPETokenizer` class.

## Interface

- `train(lines, vocab_size)`  
  Trains the BPE tokenizer on the provided text and builds the vocabulary.

- `encode(text) → list[int]`  
  Converts text into a sequence of integer token IDs.  
  **No BOS/EOS tokens are added.**

- `decode(ids) → str`  
  Converts integer token IDs back into the original text representation.

- `save(directory)`  
  Saves the trained tokenizer vocabulary and merge rules to the specified directory.

- `load(directory)`  
  Class method that loads a previously saved tokenizer from the specified directory.

In [30]:
class BPETokenizer:
    """
    Byte Pair Encoding tokenizer trained from scratch.

    Special tokens (fixed IDs):
        <pad> = 0
        <bos> = 1
        <eos> = 2
        <unk> = 3

    Usage:
        tokenizer = BPETokenizer()
        tokenizer.train(src_lines + tgt_lines, vocab_size=8000)
        ids     = tokenizer.encode("I am a student .")
        text    = tokenizer.decode(ids)
        tokenizer.save("/content/drive/MyDrive/Transformer_Mine/vocab")
        tokenizer = BPETokenizer.load("/content/drive/.../vocab")
    """
    SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>', '<unk>']
    PAD_ID = 0
    BOS_ID = 1
    EOS_ID = 2
    UNK_ID = 3
    END_OF_WORD = '</w>'

    def __init__(self):
        self.merge_rules : list= []
        self.token2id: dict = {}
        self.id2token: dict = {}
        self.vocab_size: int = 0
        self._trained: bool= False

    def train(
        self,
        lines:list,
        vocab_size : int,
        normalise_fn = normalise,
        verbose: bool = True,
    ):
        """
        Train BPE on combined source + target lines.

        Args:
            lines: list of raw strings (src + tgt combined)
            vocab_size : target vocabulary size
            normalise_fn: text normalisation function
            verbose: print training progress
        """
        merges, symbol_vocab, _ = train_bpe(
            lines= lines,
            vocab_size = vocab_size,
            normalise_fn= normalise_fn,
            verbose= verbose,
        )

        self.merge_rules = merges

        # Build vocabulary: special tokens first, then BPE symbols
        all_symbols   = self.SPECIAL_TOKENS + sorted(symbol_vocab)

        self.token2id = {s: i for i, s in enumerate(all_symbols)}

        self.id2token = {i: s for s, i in self.token2id.items()}

        self.vocab_size = len(self.token2id)

        self._trained   = True

        if verbose:
            print(f"\nTokenizer trained.")
            print(f"Vocabulary size : {self.vocab_size}")
            print(f"Merge rules: {len(self.merge_rules)}")

    # Encoding
    def encode(self, text: str, normalise_fn=normalise) -> list:
        """
        Encode a string to a list of integer token IDs.

        Does NOT add BOS/EOS — use encode_with_special() for that.

        Args:
            text: raw input string

        Returns:
            list of integer IDs
        """
        assert self._trained, "Call train() or load() before encode()."
        return encode_line(
            text= text,
            merge_rules = self.merge_rules,
            token2id = self.token2id,
            normalise_fn= normalise_fn,
            unk_id = self.UNK_ID,
        )

    def encode_with_special(self, text: str) -> list:
        """
        Encode text and wrap with BOS and EOS tokens.

        Returns:
            [BOS_ID] + encode(text) + [EOS_ID]
        """
        return [self.BOS_ID] + self.encode(text) + [self.EOS_ID]

    # Decoding
    def decode(self, ids: list) -> str:
        """
        Decode a list of token IDs back to a string.

        Skips PAD, BOS, EOS tokens automatically.

        Args:
            ids: list of integer token IDs

        Returns:
            decoded string
        """
        assert self._trained, "Call train() or load() before decode()."
        return decode_ids(
            ids = ids,
            id2token = self.id2token,
            pad_id = self.PAD_ID,
            bos_id = self.BOS_ID,
            eos_id = self.EOS_ID,
        )

    # Properties
    @property
    def pad_id(self): return self.PAD_ID

    @property
    def bos_id(self): return self.BOS_ID

    @property
    def eos_id(self): return self.EOS_ID

    @property
    def unk_id(self): return self.UNK_ID

    # Save
    def save(self, directory: str):
        """
        Save tokenizer to directory.

        Creates two files:
            vocab.json — token2id mapping
            merges.json  — ordered list of merge rules

        Args:
            directory: path to save directory (must exist)
        """
        assert self._trained, "Nothing to save — tokenizer not trained yet."
        os.makedirs(directory, exist_ok=True)

        vocab_path  = os.path.join(directory, 'vocab.json')
        merges_path = os.path.join(directory, 'merges.json')

        with open(vocab_path, 'w', encoding='utf-8') as f:
            json.dump(self.token2id, f, ensure_ascii=False, indent=2)

        with open(merges_path, 'w', encoding='utf-8') as f:
            json.dump(self.merge_rules, f, ensure_ascii=False, indent=2)

        print(f"Tokenizer saved:")
        print(f"   {vocab_path}")
        print(f"   {merges_path}")

    # Load
    @classmethod
    def load(cls, directory: str) -> 'BPETokenizer':
        """
        Load a previously saved tokenizer from directory.

        Args:
            directory: path containing vocab.json and merges.json

        Returns:
            BPETokenizer instance ready to use
        """
        vocab_path  = os.path.join(directory, 'vocab.json')
        merges_path = os.path.join(directory, 'merges.json')

        assert os.path.exists(vocab_path),  f"Not found: {vocab_path}"
        assert os.path.exists(merges_path), f"Not found: {merges_path}"

        tokenizer = cls()

        with open(vocab_path, encoding='utf-8') as f:
            tokenizer.token2id = json.load(f)
        tokenizer.id2token = {int(i): s for s, i in tokenizer.token2id.items()}

        with open(merges_path, encoding='utf-8') as f:
            raw = json.load(f)
        tokenizer.merge_rules = [tuple(pair) for pair in raw]

        tokenizer.vocab_size = len(tokenizer.token2id)
        tokenizer._trained   = True

        print(f"✅ Tokenizer loaded from {directory}")
        print(f"   Vocabulary size : {tokenizer.vocab_size}")
        print(f"   Merge rules     : {len(tokenizer.merge_rules)}")

        return tokenizer

#7 - Train on Stage A and Save

In [31]:
STAGE_A_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_a')
VOCAB_DIR_A = os.path.join(PROJECT_ROOT, 'vocab/stage_a')

src_a, tgt_a = load_raw_pairs(
    os.path.join(STAGE_A_DIR, 'train.en'),
    os.path.join(STAGE_A_DIR, 'train.de'),
)

print("Training BPE tokenizer on Stage A …")
print("=" * 50)

tokenizer_a = BPETokenizer()
tokenizer_a.train(
    lines      = src_a + tgt_a,
    vocab_size = 200,
    verbose    = True,
)

tokenizer_a.save(VOCAB_DIR_A)
print(f"\nVocabulary saved to: {VOCAB_DIR_A}")

Training BPE tokenizer on Stage A …
Initial character vocabulary size : 56
Unique word types in corpus       : 81
Target vocabulary size            : 200
Merges needed                     : max(140, 0)

  Merge    14/140: 'tu' + 'd' → 'tud'  (freq=4)
  Merge    28/140: 'li' + 'k' → 'lik'  (freq=3)
  Merge    42/140: 'ch' + 'er</w>' → 'cher</w>'  (freq=2)
  Merge    56/140: 'ca' + 't</w>' → 'cat</w>'  (freq=2)
  Merge    70/140: 'a' + 'u' → 'au'  (freq=2)
  Best pair frequency < 2 at step 77. Stopping.

✅ BPE training complete.
   Merges learned    : 77
   Symbol vocab size : 133
   Total vocab size  : 137  (+4 special tokens)

Tokenizer trained.
Vocabulary size : 137
Merge rules: 77
Tokenizer saved:
   /content/drive/MyDrive/Transformer_Mine/vocab/stage_a/vocab.json
   /content/drive/MyDrive/Transformer_Mine/vocab/stage_a/merges.json

Vocabulary saved to: /content/drive/MyDrive/Transformer_Mine/vocab/stage_a


#8 - Train on Stage B and Save

Stage B uses a larger **8,000-token vocabulary** trained on the Tatoeba dataset.

BPE training is performed on the Stage B corpus and typically takes approximately **2–5 minutes on the Colab CPU**.

Progress is printed after every **10% of the total merge operations** to monitor the training process.

In [32]:
import time

In [33]:
STAGE_B_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_b')
VOCAB_DIR_B = os.path.join(PROJECT_ROOT, 'vocab/stage_b')

src_b, tgt_b = load_raw_pairs(
    os.path.join(STAGE_B_DIR, 'train.en'),
    os.path.join(STAGE_B_DIR, 'train.de'),
)

print(f"Stage B training lines: {len(src_b) + len(tgt_b):,}")
print("Training BPE tokenizer on Stage B (vocab_size=8000) …")
print("=" * 50)

t0 = time.time()

tokenizer_b = BPETokenizer()
tokenizer_b.train(
    lines      = src_b + tgt_b,
    vocab_size = 8000,
    verbose    = True,
)

elapsed = time.time() - t0
print(f"\nTraining time: {elapsed:.1f}s")

tokenizer_b.save(VOCAB_DIR_B)
print(f"Vocabulary saved to: {VOCAB_DIR_B}")

Stage B training lines: 89,162
Training BPE tokenizer on Stage B (vocab_size=8000) …
This may take 2–5 minutes.
Initial character vocabulary size : 176
Unique word types in corpus       : 22589
Target vocabulary size            : 8000
Merges needed                     : max(7820, 0)

  Merge   782/7820: 'wir' + 'k' → 'wirk'  (freq=168)
  Merge  1564/7820: 'b' + 'eit</w>' → 'beit</w>'  (freq=72)
  Merge  2346/7820: 'b' + 'lieb</w>' → 'blieb</w>'  (freq=46)
  Merge  3128/7820: 'app' + 'ro' → 'appro'  (freq=31)
  Merge  3910/7820: 'K' + 'ünst' → 'Künst'  (freq=23)
  Merge  4692/7820: 'interes' + 'siert</w>' → 'interessiert</w>'  (freq=18)
  Merge  5474/7820: 'skin' + 'ny</w>' → 'skinny</w>'  (freq=14)
  Merge  6256/7820: 'au' + 'fri' → 'aufri'  (freq=12)
  Merge  7038/7820: 'd' + 'einer</w>' → 'deiner</w>'  (freq=10)
  Merge  7820/7820: 'res' + 'ults</w>' → 'results</w>'  (freq=8)

✅ BPE training complete.
   Merges learned    : 7820
   Symbol vocab size : 7996
   Total vocab size  : 8000

# 9 - Full Verification

Verify both trained tokenizers using the following tests:

1. **Round-trip Encode → Decode**  
   Encode known sentences into token IDs and decode them back to verify that the tokenizer can reconstruct the text correctly.

2. **Unknown Word Handling (`<unk>`)**  
   Test how the tokenizer handles words or subword sequences that are not present in the learned vocabulary.

3. **BOS/EOS Wrapping**  
   Verify that encoded sequences can be correctly wrapped with the special **Beginning-of-Sequence (BOS)** and **End-of-Sequence (EOS)** tokens.

4. **Token ID Range Check**  
   Confirm that every generated token ID is within the valid vocabulary range:

   ```text
   0 ≤ token_id < vocab_size

In [34]:
test_sents = [
    ("EN", "I am a student ."),
    ("EN", "She is a teacher ."),
    ("DE", "Ich bin ein Student ."),
    ("DE", "Sie ist eine Lehrerin ."),
]

In [35]:
print("=" * 60)
print("VERIFICATION — Stage A Tokenizer")
print("=" * 60)


VERIFICATION — Stage A Tokenizer


In [36]:
for lang, sent in test_sents:
    ids = tokenizer_a.encode(sent)
    decoded = tokenizer_a.decode(ids)
    wrapped = tokenizer_a.encode_with_special(sent)
    match = normalise(sent) == normalise(decoded)

    print(f"\n[{lang}] Input  : '{sent}'")
    print(f" IDs : {ids}")
    print(f" Decoded: '{decoded}'")
    print(f" Wrapped: {wrapped}  (BOS={BOS_ID} at start, EOS={EOS_ID} at end)")
    print(f" Match  : {'✅' if match else '❌'}")

    # ID range check
    assert all(0 <= i < tokenizer_a.vocab_size for i in wrapped), \
        f"ID out of range in: {wrapped}"

print("\n✅ ID range check passed for all examples.")



[EN] Input  : 'I am a student .'
 IDs : [17, 38, 37, 116, 118, 4]
 Decoded: 'I am a student .'
 Wrapped: [1, 17, 38, 37, 116, 118, 4, 2]  (BOS=1 at start, EOS=2 at end)
 Match  : ✅

[EN] Input  : 'She is a teacher .'
 IDs : [27, 85, 37, 119, 36, 52, 4]
 Decoded: 'She is a teacher .'
 Wrapped: [1, 27, 85, 37, 119, 36, 52, 4, 2]  (BOS=1 at start, EOS=2 at end)
 Match  : ✅

[DE] Input  : 'Ich bin ein Student .'
 IDs : [18, 44, 63, 29, 118, 4]
 Decoded: 'Ich bin ein Student .'
 Wrapped: [1, 18, 44, 63, 29, 118, 4, 2]  (BOS=1 at start, EOS=2 at end)
 Match  : ✅

[DE] Input  : 'Sie ist eine Lehrerin .'
 IDs : [28, 86, 64, 24, 60, 110, 111, 99, 4]
 Decoded: 'Sie ist eine Lehrerin .'
 Wrapped: [1, 28, 86, 64, 24, 60, 110, 111, 99, 4, 2]  (BOS=1 at start, EOS=2 at end)
 Match  : ✅

✅ ID range check passed for all examples.


In [37]:
print("\nReloading Stage A tokenizer from disk …")
tokenizer_a_reloaded = BPETokenizer.load(VOCAB_DIR_A)

for lang, sent in test_sents:
    ids_orig = tokenizer_a.encode(sent)
    ids_reloaded = tokenizer_a_reloaded.encode(sent)
    assert ids_orig == ids_reloaded, \
        f"Reload mismatch for '{sent}': {ids_orig} vs {ids_reloaded}"

print("✅ Reload verification passed — saved and loaded tokenizers are identical.")


Reloading Stage A tokenizer from disk …
✅ Tokenizer loaded from /content/drive/MyDrive/Transformer_Mine/vocab/stage_a
   Vocabulary size : 137
   Merge rules     : 77
✅ Reload verification passed — saved and loaded tokenizers are identical.


In [38]:
stage_b_tests = [
    "The government announced new economic policies .",
    "Die Regierung hat neue Wirtschaftspolitiken angekündigt .",
    "unknownxyzword123",   # should produce <unk> tokens
]


In [40]:
for sent in stage_b_tests:
    ids = tokenizer_b.encode(sent)
    decoded = tokenizer_b.decode(ids)
    has_unk = UNK_ID in ids
    print(f"\nInput : '{sent}'")
    print(f"IDs : {ids[:10]}{'...' if len(ids) > 10 else ''}")
    print(f"Decoded:'{decoded}'")
    print(f"Has UNK: {has_unk}")


Input : 'The government announced new economic policies .'
IDs : [1638, 4197, 7365, 5290, 2113, 7183, 2705, 5479, 3272, 5608]...
Decoded:'The government announced new economic policies .'
Has UNK: False

Input : 'Die Regierung hat neue Wirtschaftspolitiken angekündigt .'
IDs : [355, 1320, 4142, 3537, 4317, 5475, 1847, 7109, 1988, 6810]...
Decoded:'Die Regierung hat neue Wirtschaftspolitiken angekündigt .'
Has UNK: False

Input : 'unknownxyzword123'
IDs : [7183, 4875, 7649, 7716, 7724, 7752, 7668, 2963, 17, 19]...
Decoded:'unknownxyzword123'
Has UNK: False


# 10 - Integrate with Phase 2 Pipeline

Replace the dummy tokenizer from Phase 2 with the **real BPE tokenizer** and rebuild the `tf.data.Dataset`.

This serves as the **definitive integration test** between Phase 2 and Phase 3.

If this cell executes successfully, the **Phase 2 data pipeline and Phase 3 BPE tokenizer are fully compatible**.

In [41]:
from data_pipeline import build_tf_dataset, create_padding_mask, create_decoder_mask

In [46]:
MAX_LEN_A = 20
BATCH_SIZE_A = 4

In [47]:
encode_fn_a = tokenizer_a.encode

In [48]:
dataset_a_bpe = build_tf_dataset(
    src_lines= src_a,
    tgt_lines = tgt_a,
    encode_fn = encode_fn_a,
    max_len = MAX_LEN_A,
    batch_size = BATCH_SIZE_A,
    bos_id = tokenizer_a.bos_id,
    eos_id = tokenizer_a.eos_id,
    pad_id = tokenizer_a.pad_id,
    shuffle = True,
    shuffle_seed = 42,
)

In [49]:
print("BPE-powered dataset (Stage A):")
for enc_in, dec_in, dec_tgt in dataset_a_bpe.take(1):
    print(f"\n  encoder_input shape  : {enc_in.shape}")
    print(f"  decoder_input shape  : {dec_in.shape}")
    print(f"  decoder_target shape : {dec_tgt.shape}")

    print(f"\n  First example:")
    print(f"    encoder_input IDs : {enc_in[0].numpy().tolist()}")
    print(f"    encoder_input text: '{tokenizer_a.decode(enc_in[0].numpy())}'")
    print(f"    decoder_input IDs : {dec_in[0].numpy().tolist()}")
    print(f"    decoder_input text: '{tokenizer_a.decode(dec_in[0].numpy())}'")
    print(f"    decoder_tgt IDs   : {dec_tgt[0].numpy().tolist()}")
    print(f"    decoder_tgt text  : '{tokenizer_a.decode(dec_tgt[0].numpy())}'")

    # Masks
    enc_mask = create_padding_mask(enc_in)
    dec_mask = create_decoder_mask(dec_in, tgt_len=MAX_LEN_A)
    print(f"\n  enc_mask shape : {enc_mask.shape}")
    print(f"  dec_mask shape : {dec_mask.shape}")

    assert dec_in[0, 0].numpy() == BOS_ID, "decoder_input must start with BOS"
    print("\n✅ BOS at decoder_input position 0 confirmed.")

print("   Real BPE tokenizer is now the active tokenizer.")

BPE-powered dataset (Stage A):

  encoder_input shape  : (4, 20)
  decoder_input shape  : (4, 20)
  decoder_target shape : (4, 20)

  First example:
    encoder_input IDs : [1, 17, 92, 59, 94, 127, 47, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    encoder_input text: 'I like music .'
    decoder_input IDs : [1, 18, 97, 25, 127, 88, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    decoder_input text: 'Ich mag Musik .'
    decoder_tgt IDs   : [18, 97, 25, 127, 88, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    decoder_tgt text  : 'Ich mag Musik .'

  enc_mask shape : (4, 1, 1, 20)
  dec_mask shape : (4, 1, 20, 20)

✅ BOS at decoder_input position 0 confirmed.
   Real BPE tokenizer is now the active tokenizer.


# 11 - Save BPETokenizer Class to Drive

Save the `BPETokenizer` source code to Google Drive so it can be imported and reused in any future Colab session without rerunning the tokenizer implementation cells.

In [50]:
TOKENIZER_PY = os.path.join(PROJECT_ROOT, 'tokenizer.py')

code = open.__doc__

In [56]:
with open(TOKENIZER_PY, 'w', encoding='utf-8') as f:
    f.write('''"""
tokenizer.py — BPE Tokenizer for Transformer Reproduction Project
Phase 3: Byte Pair Encoding from scratch.

Usage:
    import sys
    sys.path.insert(0, "/content/drive/MyDrive/transformer_reproduction")
    from tokenizer import BPETokenizer, normalise

    tokenizer = BPETokenizer.load("vocab/stage_b")
    ids  = tokenizer.encode("I am a student .")
    text = tokenizer.decode(ids)
"""

import re, json, os, collections

END_OF_WORD    = "</w>"
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3

def normalise(text, lowercase=False):
    if lowercase:
        text = text.lower()
    text = re.sub(r\'([\\.,!?;:\\(\\)\\[\\]{}\\"\\'\\-/])\', r\' \\1 \', text)
    text = re.sub(r\'\\s+\', \' \', text)
    return text.strip()

def text_to_word_freqs(lines, normalise_fn=normalise):
    freq = collections.Counter()
    for line in lines:
        for word in normalise_fn(line).split():
            chars = tuple(list(word[:-1]) + [word[-1] + END_OF_WORD])
            freq[chars] += 1
    return freq

def get_pair_freqs(word_freqs):
    pairs = collections.Counter()
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i+1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    new_freqs = collections.Counter()
    a, b = pair
    merged = a + b
    for word, freq in word_freqs.items():
        new_word, i = [], 0
        while i < len(word):
            if i < len(word)-1 and word[i] == a and word[i+1] == b:
                new_word.append(merged); i += 2
            else:
                new_word.append(word[i]); i += 1
        new_freqs[tuple(new_word)] += freq
    return new_freqs

def train_bpe(lines, vocab_size, normalise_fn=normalise, verbose=True):
    word_freqs  = text_to_word_freqs(lines, normalise_fn)
    base_vocab  = {s for word in word_freqs for s in word}
    merges, vocab = [], set(base_vocab)
    target = max(0, vocab_size - len(SPECIAL_TOKENS) - len(base_vocab))
    for i in range(target):
        pf = get_pair_freqs(word_freqs)
        if not pf or max(pf.values()) < 2: break
        best = max(pf, key=pf.get)
        merges.append(best); vocab.add(best[0]+best[1])
        word_freqs = merge_pair(best, word_freqs)
        if verbose and (i+1) % max(1, target//10) == 0:
            print(f"  Merge {i+1}/{target}: {best[0]}+{best[1]}")
    return merges, vocab, word_freqs

def encode_word(word, merge_rules):
    if not word: return []
    symbols = list(word[:-1]) + [word[-1] + END_OF_WORD]
    for (a, b) in merge_rules:
        merged, i, new = a+b, 0, []
        while i < len(symbols):
            if i < len(symbols)-1 and symbols[i]==a and symbols[i+1]==b:
                new.append(merged); i += 2
            else:
                new.append(symbols[i]); i += 1
        symbols = new
    return symbols

def encode_line(text, merge_rules, token2id, normalise_fn=normalise, unk_id=UNK_ID):
    return [token2id.get(sw, unk_id)
            for word in normalise_fn(text).split()
            for sw in encode_word(word, merge_rules)]

def decode_ids(ids, id2token, pad_id=PAD_ID, bos_id=BOS_ID, eos_id=EOS_ID):
    skip = {pad_id, bos_id, eos_id}
    text = "".join(id2token.get(i, "<unk>") for i in ids if i not in skip)
    return text.replace(END_OF_WORD, " ").strip()

class BPETokenizer:
    SPECIAL_TOKENS = SPECIAL_TOKENS
    PAD_ID=0; BOS_ID=1; EOS_ID=2; UNK_ID=3
    END_OF_WORD = END_OF_WORD

    def __init__(self):
        self.merge_rules=[]; self.token2id={}; self.id2token={}
        self.vocab_size=0; self._trained=False

    def train(self, lines, vocab_size, normalise_fn=normalise, verbose=True):
        merges, sym_vocab, _ = train_bpe(lines, vocab_size, normalise_fn, verbose)
        self.merge_rules = merges
        all_syms = self.SPECIAL_TOKENS + sorted(sym_vocab)
        self.token2id = {s:i for i,s in enumerate(all_syms)}
        self.id2token = {i:s for s,i in self.token2id.items()}
        self.vocab_size = len(self.token2id); self._trained = True

    def encode(self, text, normalise_fn=normalise):
        return encode_line(text, self.merge_rules, self.token2id, normalise_fn, self.UNK_ID)

    def encode_with_special(self, text):
        return [self.BOS_ID] + self.encode(text) + [self.EOS_ID]

    def decode(self, ids):
        return decode_ids(ids, self.id2token, self.PAD_ID, self.BOS_ID, self.EOS_ID)

    @property
    def pad_id(self): return self.PAD_ID
    @property
    def bos_id(self): return self.BOS_ID
    @property
    def eos_id(self): return self.EOS_ID
    @property
    def unk_id(self): return self.UNK_ID

    def save(self, directory):
        os.makedirs(directory, exist_ok=True)
        with open(os.path.join(directory,"vocab.json"),"w",encoding="utf-8") as f:
            json.dump(self.token2id, f, ensure_ascii=False, indent=2)
        with open(os.path.join(directory,"merges.json"),"w",encoding="utf-8") as f:
            json.dump(self.merge_rules, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, directory):
        t = cls()
        with open(os.path.join(directory,"vocab.json"),encoding="utf-8") as f:
            t.token2id = json.load(f)
        t.id2token = {int(i):s for s,i in t.token2id.items()}
        with open(os.path.join(directory,"merges.json"),encoding="utf-8") as f:
            t.merge_rules = [tuple(p) for p in json.load(f)]
        t.vocab_size = len(t.token2id); t._trained = True
        return t
''')

print(f"DONE tokenizer.py saved to:")
print(f"   {TOKENIZER_PY}")
print()
print("To use in any future session:")
print(" import sys")
print(f' sys.path.insert(0, "{PROJECT_ROOT}")')
print(" from tokenizer import BPETokenizer")
print(" tokenizer = BPETokenizer.load('vocab/stage_b')")

DONE tokenizer.py saved to:
   /content/drive/MyDrive/Transformer_Mine/tokenizer.py

To use in any future session:
 import sys
 sys.path.insert(0, "/content/drive/MyDrive/Transformer_Mine")
 from tokenizer import BPETokenizer
 tokenizer = BPETokenizer.load('vocab/stage_b')
